## The city of Spring View is taking bids from six bus companies on the eight routes that must be driven in the surrounding school district. Each company enters a bid of how much it will charge to drive selected routes, although not all companies bid on all routes. (If a company does not bid on a route, the corresponding entry is blank.) The city must decide which companies to assign to which routes with the specifications that (1) if a company does not bid on a route, it cannot be assigned to that route; (2) exactly one company must be assigned to each route; and (3) a company can be assigned to at most two routes. The objective is to minimize the total cost of covering all routes.


In [ ]:
import gurobipy as gp
from gurobipy import GRB

In [ ]:
# Sets
companies = range(1, 7)   # Companies 1–6
routes = range(1, 9)      # Routes 1–8

# Cost data: only include bids that actually exist
cost = {
    (1, 3): 7800,
    (2, 6): 3300,
    (2, 7): 4900,
    (3, 2): 4800,
    (5, 4): 3900,
    (5, 8): 3000,
    (6, 1): 7000,
    (6, 5): 5600
}

In [ ]:
# Create Model

model = gp.Model("Spring_View_Bus_Assignment")

In [ ]:
# Create Decision Variables

x = model.addVars(
    cost.keys(),
    vtype=GRB.BINARY,
    name="assign"
)

In [ ]:
# Objective Function

model.setObjective(
    gp.quicksum(cost[i, j] * x[i, j] for i, j in cost),
    GRB.MINIMIZE
)

In [ ]:
# Constraint: exactly one company per route

for j in routes:
    model.addConstr(
        gp.quicksum(x[i, j] for (i, r) in cost if r == j) == 1,
        name=f"route_{j}_coverage"
    )

In [ ]:
# Constraint: each company can do at most two routes

for i in companies:
    model.addConstr(
        gp.quicksum(x[i, j] for (c, j) in cost if c == i) <= 2,
        name=f"company_{i}_capacity"
    )

In [ ]:
# Solve

model.optimize()

In [ ]:
# ouput

if model.status == GRB.OPTIMAL:
    print(f"Optimal total cost: ${model.objVal:,.0f}\n")

    for (i, j) in x:
        if x[i, j].x > 0.5:
            print(f"Company {i} assigned to Route {j} at cost ${cost[i, j]:,}")
